# Partition 1 EDA

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Extracting partition 1

In [2]:
import os

# Where the archive currently lives
drive_data_dir = '/content/drive/MyDrive/solar_flare_forecasting/Data'
# True high-speed local Colab SSD path
local_extract_dir = '/content/solar_flare_data'

# Ensure the local extraction directory exists
os.makedirs(local_extract_dir, exist_ok=True)

files_to_extract = {
    "partition1_instances.tar.gz": "https://dataverse.harvard.edu/api/access/datafile/:persistentId?persistentId=doi:10.7910/DVN/EBCFKM/BMXYCB"
}

print("\n--- Starting High-Speed Local Extraction ---")

for file_name in files_to_extract.keys():
    archive_path = os.path.join(drive_data_dir, file_name)

    if os.path.exists(archive_path):
        print(f"⚡ Unpacking {file_name} into Colab SSD ({local_extract_dir})...")
        # -xzf: extract gzipped file, -C: target local directory
        # Using native system tar is indeed the fastest method
        exit_code = os.system(f"tar -xzf {archive_path} -C {local_extract_dir}")

        if exit_code == 0:
            print(f"✅ Successfully unpacked {file_name}!")
        else:
            print(f"❌ Error occurred while unpacking {file_name}. Exit code: {exit_code}")
    else:
        print(f"⚠️ Could not find archive at {archive_path}, skipping extraction.")

print(f"\n🎉 All processes complete! Data is ready in '{local_extract_dir}'.")


--- Starting High-Speed Local Extraction ---
⚡ Unpacking partition1_instances.tar.gz into Colab SSD (/content/solar_flare_data)...
✅ Successfully unpacked partition1_instances.tar.gz!

🎉 All processes complete! Data is ready in '/content/solar_flare_data'.


## Exploring all files in Partition 1.

for missing data and time stamp differences


### Exploration loop

In [3]:
import pandas as pd
import numpy as np
import glob
from collections import Counter

fl_files = glob.glob("/content/solar_flare_data/partition1/FL/*.csv")
nf_files = glob.glob("/content/solar_flare_data/partition1/NF/*.csv")
all_files = fl_files + nf_files

print(f"Total files found in Partition 1: {len(all_files)} ({len(fl_files)} FL, {len(nf_files)} NF)")

# Reset metrics every time the cell is run
total_rows_checked = 0
total_missing_cells = 0
total_missing_rows = 0
files_with_missing_values = 0
cadence_differences = Counter()
unexpected_row_counts = []

# --- CONFIGURATION ---
SAMPLE_LIMIT = 200
files_to_process = all_files[:SAMPLE_LIMIT] if SAMPLE_LIMIT else all_files

print(f"Processing a sample of {len(files_to_process)} files...")

for path in files_to_process:
    # FIX: Added sep='\t' because the error shows the columns are tab-separated
    df = pd.read_csv(path, sep='\t')

    num_rows = len(df)
    total_rows_checked += num_rows

    if num_rows != 60:
        unexpected_row_counts.append((path.split('/')[-1], num_rows))

    null_cells = df.isnull().sum().sum()
    if null_cells > 0:
        total_missing_cells += null_cells
        files_with_missing_values += 1
        total_missing_rows += df.isnull().any(axis=1).sum()

    # Use case-insensitive check for timestamp column
    ts_col = [c for c in df.columns if 'timestamp' in c.lower()]
    if ts_col:
        try:
            # Ensure we are parsing the column correctly
            timestamps = pd.to_datetime(df[ts_col[0]], errors='coerce')
            time_deltas = timestamps.diff().dropna()
            delta_minutes = time_deltas.dt.total_seconds() / 60.0
            for delta in delta_minutes:
                cadence_differences[delta] += 1
        except Exception as e:
            print(f"Could not parse timestamps in {path}: {e}")

print("\n--- EXPLORATION COMPLETED ---")

Total files found in Partition 1: 73492 (1254 FL, 72238 NF)
Processing a sample of 200 files...

--- EXPLORATION COMPLETED ---


### Report

In [4]:
print("==================================================")
print("             DATA QUALITY ANALYSIS REPORT         ")
print("==================================================")

# Calculations
missing_cell_pct = (total_missing_cells / (total_rows_checked * 55)) * 100 if total_rows_checked > 0 else 0
missing_row_pct = (total_missing_rows / total_rows_checked) * 100 if total_rows_checked > 0 else 0

# 1. Missing Value Report
print(f"• Total Rows Scanned:      {total_rows_checked:,}")
print(f"• Files with Null Values:   {files_with_missing_values} out of {len(files_to_process)}")
print(f"• Total Missing Data Cells: {total_missing_cells:,} ({missing_cell_pct:.2f}% of all individual cells)")
print(f"• Total Rows with Missing:  {total_missing_rows:,} ({missing_row_pct:.2f}% of all scanned rows) ⚠️")

print("\n--------------------------------------------------")
print("• Row Count Deviations (Expected: 60 rows per file):")
if len(unexpected_row_counts) == 0:
    print("  -> Excellent! Every scanned file has exactly 60 rows.")
else:
    print(f"  -> Found {len(unexpected_row_counts)} files with non-60 row limits:")
    for file_name, rows in unexpected_row_counts[:10]:
        print(f"     - {file_name}: {rows} rows")

print("\n--------------------------------------------------")
print("• Timestamp Difference / Cadence Distribution:")
if not cadence_differences:
    print("  -> No timestamp cadence could be computed.")
else:
    print("  Expected step size is 12.0 minutes.")
    for minutes, count in sorted(cadence_differences.items()):
        status = " (Expected Cadence)" if minutes == 12.0 else " ⚠️ (Cadence Gap/Jump)"
        print(f"     - Gap of {minutes} minutes: {count:,} occurrences{status}")
print("==================================================")

             DATA QUALITY ANALYSIS REPORT         
• Total Rows Scanned:      12,000
• Files with Null Values:   200 out of 200
• Total Missing Data Cells: 95,306 (14.44% of all individual cells)
• Total Rows with Missing:  12,000 (100.00% of all scanned rows) ⚠️

--------------------------------------------------
• Row Count Deviations (Expected: 60 rows per file):
  -> Excellent! Every scanned file has exactly 60 rows.

--------------------------------------------------
• Timestamp Difference / Cadence Distribution:
  Expected step size is 12.0 minutes.
     - Gap of 12.0 minutes: 11,800 occurrences (Expected Cadence)


### Exploration excluding the Label columns

`BFLARE_LABEL`,
 `CFLARE_LABEL`,
 `MFLARE_LABEL`,
 `XFLARE_LABEL`,
 `BFLARE_LABEL_LOC`,
 `CFLARE_LABEL_LOC`,
 `MFLARE_LABEL_LOC`,
 `XFLARE_LABEL_LOC`

 They are empty most of the time as Flares are the minority class.

In [5]:
import pandas as pd
import numpy as np
import glob
from collections import Counter

# Define the columns to exclude
COLS_TO_EXCLUDE = [
    'BFLARE_LABEL', 'CFLARE_LABEL', 'MFLARE_LABEL', 'XFLARE_LABEL',
    'BFLARE_LABEL_LOC', 'CFLARE_LABEL_LOC', 'MFLARE_LABEL_LOC', 'XFLARE_LABEL_LOC'
]

fl_files = glob.glob("/content/solar_flare_data/partition1/FL/*.csv")
nf_files = glob.glob("/content/solar_flare_data/partition1/NF/*.csv")
all_files = fl_files + nf_files

# Reset metrics
total_rows_checked = 0
total_missing_cells = 0
total_missing_rows = 0
files_with_missing_values = 0
cadence_differences = Counter()
unexpected_row_counts = []

SAMPLE_LIMIT = 200
files_to_process = all_files[:SAMPLE_LIMIT] if SAMPLE_LIMIT else all_files

print(f"Processing {len(files_to_process)} files (Excluding {len(COLS_TO_EXCLUDE)} label columns)...")

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')

    # Drop the labels so they don't count towards missing data metrics
    df_features = df.drop(columns=[c for c in COLS_TO_EXCLUDE if c in df.columns])

    num_rows = len(df_features)
    total_rows_checked += num_rows

    if num_rows != 60:
        unexpected_row_counts.append((path.split('/')[-1], num_rows))

    null_cells = df_features.isnull().sum().sum()
    if null_cells > 0:
        total_missing_cells += null_cells
        files_with_missing_values += 1
        total_missing_rows += df_features.isnull().any(axis=1).sum()

    ts_col = [c for c in df.columns if 'timestamp' in c.lower()]
    if ts_col:
        timestamps = pd.to_datetime(df[ts_col[0]], errors='coerce')
        time_deltas = timestamps.diff().dropna()
        delta_minutes = time_deltas.dt.total_seconds() / 60.0
        for delta in delta_minutes:
            cadence_differences[delta] += 1

# Update report variables for the next cell to use
num_features_checked = df_features.shape[1]
print(f"\n--- EXPLORATION COMPLETED ({num_features_checked} features analyzed) ---")

Processing 200 files (Excluding 8 label columns)...

--- EXPLORATION COMPLETED (47 features analyzed) ---


### Report

Excluding Sparse Label Columns

In [6]:
print("==================================================")
print("        DATA QUALITY ANALYSIS REPORT       ")
print("      (Excluding Sparse Label Columns)           ")
print("==================================================")

# Dynamic feature count from previous cell
features_count = num_features_checked if 'num_features_checked' in locals() else 47

# Calculations
missing_cell_pct = (total_missing_cells / (total_rows_checked * features_count)) * 100 if total_rows_checked > 0 else 0
missing_row_pct = (total_missing_rows / total_rows_checked) * 100 if total_rows_checked > 0 else 0

# 1. Missing Value Report
print(f"• Total Rows Scanned:      {total_rows_checked:,}")
print(f"• Features per Row:        {features_count}")
print(f"• Files with Null Values:   {files_with_missing_values} out of {len(files_to_process)}")
print(f"• Total Missing Data Cells: {total_missing_cells:,} ({missing_cell_pct:.4f}% of total cells)")
print(f"• Total Rows with Missing:  {total_missing_rows:,} ({missing_row_pct:.2f}% of scanned rows)")

print("\n--------------------------------------------------")
print("• Row Count Deviations (Expected: 60 rows):")
if not unexpected_row_counts:
    print("  -> All files have exactly 60 rows.")
else:
    print(f"  -> Found {len(unexpected_row_counts)} anomalies.")

print("\n--------------------------------------------------")
print("• Cadence Distribution:")
for minutes, count in sorted(cadence_differences.items()):
    status = " (Standard)" if minutes == 12.0 else " ⚠️ (Irregular)"
    print(f"     - {minutes} min: {count:,} times{status}")
print("==================================================")

        DATA QUALITY ANALYSIS REPORT       
      (Excluding Sparse Label Columns)           
• Total Rows Scanned:      12,000
• Features per Row:        47
• Files with Null Values:   9 out of 200
• Total Missing Data Cells: 578 (0.1025% of total cells)
• Total Rows with Missing:  17 (0.14% of scanned rows)

--------------------------------------------------
• Row Count Deviations (Expected: 60 rows):
  -> All files have exactly 60 rows.

--------------------------------------------------
• Cadence Distribution:
     - 12.0 min: 11,800 times (Standard)


In [7]:
# Set the maximum number of columns to display to None (unlimited)
pd.set_option('display.max_columns', None)

In [8]:
missing_rows_list = []
files_affected = []

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')
    df_features = df.drop(columns=[c for c in COLS_TO_EXCLUDE if c in df.columns])

    mask = df_features.isnull().any(axis=1)
    if mask.any():
        filename = path.split('/')[-1]
        files_affected.append(filename)
        rows = df_features[mask].copy()
        rows['source_file'] = filename
        missing_rows_list.append(rows)

if missing_rows_list:
    all_missing_df = pd.concat(missing_rows_list, ignore_index=True)
    print(f"✅ Total rows with missing data: {len(all_missing_df)}")
    print(f"📂 Affected files ({len(files_affected)} total): {files_affected}")

    # FIX: Arguments must be in pairs (pattern, value)
    with pd.option_context('display.max_rows', None):
        display(all_missing_df)
else:
    print("No missing data found in the current sample subset.")

✅ Total rows with missing data: 17
📂 Affected files (9 total): ['M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e2011-09-25T06:24:00.csv', 'X5.4@3386:Primary_ar1449_s2012-03-06T00:24:00_e2012-03-06T12:12:00.csv', 'X5.4@3386:Primary_ar1449_s2012-03-06T02:24:00_e2012-03-06T14:12:00.csv', 'M2.9@516:Primary_ar211_s2010-10-15T17:12:00_e2010-10-16T05:00:00.csv', 'M8.7@3206:Primary_ar1321_s2012-01-21T22:24:00_e2012-01-22T10:12:00.csv', 'X5.4@3386:Primary_ar1449_s2012-03-06T01:24:00_e2012-03-06T13:12:00.csv', 'M2.9@516:Primary_ar211_s2010-10-15T19:12:00_e2010-10-16T07:00:00.csv', 'X5.4@3386:Primary_ar1449_s2012-03-05T19:24:00_e2012-03-06T07:12:00.csv', 'M2.9@516:Primary_ar211_s2010-10-15T12:12:00_e2010-10-16T00:00:00.csv']


,Timestamp,TOTUSJH,TOTBSQ,TOTPOT,TOTUSJZ,ABSNJZH,SAVNCPP,USFLUX,TOTFZ,MEANPOT,EPSZ,MEANSHR,SHRGT45,MEANGAM,MEANGBT,MEANGBZ,MEANGBH,MEANJZH,TOTFY,MEANJZD,MEANALP,TOTFX,EPSY,EPSX,R_VALUE,CRVAL1,CRLN_OBS,CRLT_OBS,CRVAL2,HC_ANGLE,SPEI,LAT_MIN,LON_MIN,LAT_MAX,LON_MAX,QUALITY,BFLARE,CFLARE,MFLARE,XFLARE,BFLARE_LOC,CFLARE_LOC,MFLARE_LOC,XFLARE_LOC,XR_MAX,XR_QUAL,IS_TMFI,source_file
0,2011-09-25 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-9.999900e+04,0,False,M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e...
1,2011-09-25 06:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-9.999900e+04,0,False,M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e...
2,2011-09-25 06:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-9.999900e+04,0,False,M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e...
3,2012-03-06 07:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.197900e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T00:24:00_...
4,2012-03-06 07:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.782500e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T00:24:00_...
5,2012-03-06 07:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.197900e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T02:24:00_...
6,2012-03-06 07:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.782500e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T02:24:00_...
7,2010-10-15 20:36:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.897700e-08,12,False,M2.9@516:Primary_ar211_s2010-10-15T17:12:00_e2...
8,2010-10-15 20:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.023200e-07,12,False,M2.9@516:Primary_ar211_s2010-10-15T17:12:00_e2...
9,2012-01-22 10:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.614900e-07,12,False,M8.7@3206:Primary_ar1321_s2012-01-21T22:24:00_...


In [9]:
all_missing_df

,Timestamp,TOTUSJH,TOTBSQ,TOTPOT,TOTUSJZ,ABSNJZH,SAVNCPP,USFLUX,TOTFZ,MEANPOT,EPSZ,MEANSHR,SHRGT45,MEANGAM,MEANGBT,MEANGBZ,MEANGBH,MEANJZH,TOTFY,MEANJZD,MEANALP,TOTFX,EPSY,EPSX,R_VALUE,CRVAL1,CRLN_OBS,CRLT_OBS,CRVAL2,HC_ANGLE,SPEI,LAT_MIN,LON_MIN,LAT_MAX,LON_MAX,QUALITY,BFLARE,CFLARE,MFLARE,XFLARE,BFLARE_LOC,CFLARE_LOC,MFLARE_LOC,XFLARE_LOC,XR_MAX,XR_QUAL,IS_TMFI,source_file
0,2011-09-25 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-9.999900e+04,0,False,M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e...
1,2011-09-25 06:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-9.999900e+04,0,False,M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e...
2,2011-09-25 06:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-9.999900e+04,0,False,M4.0@2528:Primary_ar892_s2011-09-24T18:36:00_e...
3,2012-03-06 07:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.197900e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T00:24:00_...
4,2012-03-06 07:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.782500e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T00:24:00_...
5,2012-03-06 07:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.197900e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T02:24:00_...
6,2012-03-06 07:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.782500e-06,12,False,X5.4@3386:Primary_ar1449_s2012-03-06T02:24:00_...
7,2010-10-15 20:36:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.897700e-08,12,False,M2.9@516:Primary_ar211_s2010-10-15T17:12:00_e2...
8,2010-10-15 20:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.023200e-07,12,False,M2.9@516:Primary_ar211_s2010-10-15T17:12:00_e2...
9,2012-01-22 10:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.614900e-07,12,False,M8.7@3206:Primary_ar1321_s2012-01-21T22:24:00_...


## Analyzing Quality of rows

Based on data quality parameters:

`QUALITY`, `IS_TMFI`, `SPEI`, `XRQUALITY`

| Column     | Meaning |
|------------|---------|
| `QUALITY`  | Bitmask flag from HMI pipeline. **0 = good data**. Non-zero = something went wrong. |
| `IS_TMFI`  | "Trusted Magnetic Field Information" boolean. **`True` = trustworthy** (`QUALITY = 0` and within 70° of disk center). |
| `SPEI`     | Boolean — **`True`** means SDO was in Earth's shadow (data blackout); exclude these rows. |
| `XRQUALITY`| Quality flag for X-ray flux data — indicates how many of the 1-minute readings in the window were valid. |

### Value count loop

In [10]:
from collections import Counter

# Initialize counters for each quality column
quality_stats = {
    'QUALITY': Counter(),
    'IS_TMFI': Counter(),
    'SPEI': Counter(),
    'XR_QUAL': Counter()
}

print(f"Analyzing quality flags across {len(files_to_process)} files...")

for path in files_to_process:
    df = pd.read_csv(path, sep='\t')

    for col in quality_stats.keys():
        if col in df.columns:
            # Update counter with values from this file
            quality_stats[col].update(df[col].fillna('NaN').tolist())
        else:
            # Track missing columns if they don't exist in some files
            quality_stats[col]['COLUMN_MISSING'] += len(df)

'''
print("\n--- Global Quality Flag Counts (Sample of 200 Files) ---")
for col, counts in quality_stats.items():
    print(f"\nDistribution for {col}:")
    # Fix: Convert keys to strings to avoid TypeError during sort_index when mixing str and float
    stats_ser = pd.Series(counts)
    stats_ser.index = stats_ser.index.astype(str)
    display(stats_ser.sort_index())
'''

Analyzing quality flags across 200 files...


'\nprint("\n--- Global Quality Flag Counts (Sample of 200 Files) ---")\nfor col, counts in quality_stats.items():\n    print(f"\nDistribution for {col}:")\n    # Fix: Convert keys to strings to avoid TypeError during sort_index when mixing str and float\n    stats_ser = pd.Series(counts)\n    stats_ser.index = stats_ser.index.astype(str)\n    display(stats_ser.sort_index())\n'

### Report

In [11]:
print("==================================================")
print("         GLOBAL QUALITY FLAG REPORT              ")
print("      (Sample of 200 Files / 12,000 Rows)        ")
print("==================================================")

def print_stat_group(name, counter_dict, description):
    print(f"\n• {name}:")
    print(f"  ({description})")
    # Sort keys by string to ensure NaN and numbers don't conflict
    sorted_keys = sorted(counter_dict.keys(), key=lambda x: str(x))
    for key in sorted_keys:
        count = counter_dict[key]
        pct = (count / total_rows_checked) * 100
        print(f"    - {key}: {count:,} rows ({pct:.2f}%)")

# 1. HMI Quality Bitmask
print_stat_group(
    "HMI QUALITY FLAGS",
    quality_stats['QUALITY'],
    "0 = Good data, Non-zero = Error bitmask"
)

# 2. Trusted Magnetic Field Information
print_stat_group(
    "IS_TMFI (Trusted Magnetic Field)",
    quality_stats['IS_TMFI'],
    "True = Trustworthy, False = Potential issues"
)

# 3. SDO Earth Shadow (SPEI)
print_stat_group(
    "SPEI (Earth Shadow Indicator)",
    quality_stats['SPEI'],
    "True = Normal, False = SDO in Earth's shadow (Blackout)"
)

# 4. X-Ray Quality
print_stat_group(
    "XR_QUAL (X-Ray Flux Quality)",
    quality_stats['XR_QUAL'],
    "Count of valid 1-min readings (12 = Perfect)"
)

print("\n==================================================")

         GLOBAL QUALITY FLAG REPORT              
      (Sample of 200 Files / 12,000 Rows)        

• HMI QUALITY FLAGS:
  (0 = Good data, Non-zero = Error bitmask)
    - 0.0: 11,440 rows (95.33%)
    - 1024.0: 11 rows (0.09%)
    - 4096.0: 62 rows (0.52%)
    - 65536.0: 4 rows (0.03%)
    - 66560.0: 458 rows (3.82%)
    - 68608.0: 4 rows (0.03%)
    - 69632.0: 4 rows (0.03%)
    - NaN: 17 rows (0.14%)

• IS_TMFI (Trusted Magnetic Field):
  (True = Trustworthy, False = Potential issues)
    - False: 564 rows (4.70%)
    - True: 11,436 rows (95.30%)

• SPEI (Earth Shadow Indicator):
  (True = Normal, False = SDO in Earth's shadow (Blackout))
    - False: 21 rows (0.18%)
    - True: 11,979 rows (99.83%)

• XR_QUAL (X-Ray Flux Quality):
  (Count of valid 1-min readings (12 = Perfect))
    - 0: 152 rows (1.27%)
    - 1: 6 rows (0.05%)
    - 10: 12 rows (0.10%)
    - 11: 12 rows (0.10%)
    - 12: 11,733 rows (97.78%)
    - 2: 8 rows (0.07%)
    - 3: 8 rows (0.07%)
    - 4: 13 rows (0.11%)


## Making dataframe with rows that "fail" the GLOBAL QUALITY FLAG REPORT

`QUALITY` != 0.0 <br>
`IS_TMFI` == false <br>
`SPEI` == false <br>
`XR_QUAL` != 12 <br>

In [12]:
any_bad_quality_rows = []

# Using the sample of 200 files
for path in files_to_process:
    df_temp = pd.read_csv(path, sep='\t')

    # Use OR (|) to find rows where ANY condition is met
    mask = (
        (df_temp['QUALITY'] != 0.0) |
        (df_temp['IS_TMFI'] == False) |
        (df_temp['SPEI'] == False) |
        (df_temp['XR_QUAL'] != 12)
    )

    if mask.any():
        match = df_temp[mask].copy()
        match['source_file'] = path.split('/')[-1]
        any_bad_quality_rows.append(match)

if any_bad_quality_rows:
    any_worst_df = pd.concat(any_bad_quality_rows, ignore_index=True)
    print(f"Found {len(any_worst_df)} rows meeting at least one 'bad quality' criterion.")
    #display(any_worst_df.head())
else:
    print("No rows in the sample met any of the bad quality criteria.")

Found 753 rows meeting at least one 'bad quality' criterion.


In [13]:
any_worst_df

,Timestamp,TOTUSJH,TOTBSQ,TOTPOT,TOTUSJZ,ABSNJZH,SAVNCPP,USFLUX,TOTFZ,MEANPOT,EPSZ,MEANSHR,SHRGT45,MEANGAM,MEANGBT,MEANGBZ,MEANGBH,MEANJZH,TOTFY,MEANJZD,MEANALP,TOTFX,EPSY,EPSX,R_VALUE,CRVAL1,CRLN_OBS,CRLT_OBS,CRVAL2,HC_ANGLE,SPEI,LAT_MIN,LON_MIN,LAT_MAX,LON_MAX,QUALITY,BFLARE,BFLARE_LABEL,CFLARE,CFLARE_LABEL,MFLARE,MFLARE_LABEL,XFLARE,XFLARE_LABEL,BFLARE_LOC,BFLARE_LABEL_LOC,CFLARE_LOC,CFLARE_LABEL_LOC,MFLARE_LOC,MFLARE_LABEL_LOC,XFLARE_LOC,XFLARE_LABEL_LOC,XR_MAX,XR_QUAL,IS_TMFI,source_file
0,2011-09-27 18:00:00,3655.749194,6.690769e+10,1.551164e+24,6.469057e+13,353.656612,9.597134e+12,3.807647e+22,-5.620467e+23,20102.684000,-0.006325,53.987921,63.021927,61.070242,82.595761,91.050653,61.402949,-0.006087,2.179824e+24,0.270021,-0.010505,2.702956e+24,-0.024532,-0.030419,4.796945,282.301697,293.509796,6.877559,14.83885,13.718311,True,7.867940,-24.883097,21.626505,6.279463,66560.0,0.0,NaN,1.0,C1.1@2542:Non-verified,0.0,NaN,0.0,NaN,0.0,NaN,1.0,C1.1@2542:Non-verified,0.0,NaN,0.0,NaN,4.565400e-07,12,False,M1.2@2550:Primary_ar892_s2011-09-27T11:36:00_e...
1,2011-09-27 18:12:00,3663.933447,6.729254e+10,1.559175e+24,6.473666e+13,356.871164,9.353434e+12,3.811475e+22,-5.954640e+23,20321.226476,-0.006663,54.099266,63.075086,61.146741,82.891556,91.606555,61.798239,-0.006177,2.172109e+24,0.233555,-0.010536,2.687042e+24,-0.024305,-0.030067,4.801630,282.303619,293.399323,6.877910,14.83885,13.627030,True,7.869699,-24.773590,21.627817,6.404327,66560.0,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,1.102500e-06,12,False,M1.2@2550:Primary_ar892_s2011-09-27T11:36:00_e...
2,2011-09-27 19:12:00,3695.717637,6.835384e+10,1.588394e+24,6.456301e+13,369.412135,1.186830e+13,3.808278e+22,-5.259977e+23,21318.633433,-0.005794,54.242544,62.829082,61.179737,83.646562,93.548635,62.063772,-0.006585,2.023460e+24,0.203397,-0.010747,2.404659e+24,-0.022290,-0.026490,4.795388,282.313171,292.846924,6.879656,14.83885,13.175449,True,7.851651,-24.251707,21.641275,6.931295,66560.0,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,5.100100e-07,12,False,M1.2@2550:Primary_ar892_s2011-09-27T11:36:00_e...
3,2011-09-27 19:24:00,3765.233911,6.865048e+10,1.592155e+24,6.550008e+13,353.162238,1.152856e+13,3.839645e+22,-6.880714e+23,21137.781791,-0.007547,53.789864,62.060758,60.873298,84.068025,94.266884,61.963938,-0.006227,1.941186e+24,0.170699,-0.010212,2.434928e+24,-0.021292,-0.026707,4.819277,282.315094,292.736450,6.879990,14.83885,13.086143,True,7.829802,-24.146969,21.644341,7.054764,66560.0,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,5.390500e-07,12,False,M1.2@2550:Primary_ar892_s2011-09-27T11:36:00_e...
4,2011-04-21 00:12:00,2244.286824,3.084739e+10,3.191371e+23,4.271437e+13,285.401999,1.085084e+13,4.374446e+22,-2.104215e+25,3668.065981,-0.513640,27.072321,13.835422,33.789273,83.965280,89.935522,38.470449,0.004356,-3.855102e+23,0.076699,0.012225,5.095630e+24,0.009410,-0.124385,5.218060,186.823990,245.716537,-5.162886,-21.00230,60.196435,True,-29.260406,-88.377426,-13.792387,-29.788870,4096.0,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,8.174800e-07,12,False,M1.8@1674:Primary_ar514_s2011-04-20T23:12:00_e...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
748,2010-10-15 20:48:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,0.0,NaN,1.023200e-07,12,False,M2.9@516:Primary_ar211_s2010-10-15T12:12:00_e2...
749,2011-10-01 18:00:00,2816.578389,4.876139e+10,9.061226e+23,5.473064e+13,761.731826,3.366934e+13,3.833137e+22,-1.263337e+25,12193.700273,-0.195088,42.480964,41.029399,48.454862,91.883293,97.167220,52.729293,-0.013613,6.489916e+23,0.100039,-0.026143,-5.280842e+24,-0.010022,0.081548,4.

Confirm the 24 NaN-quality rows are exactly the 24 SPEI=False rows

In [14]:
# Filter the previously combined missing data dataframe
nan_quality_rows = any_worst_df[any_worst_df['QUALITY'].isna()]
spei_false_rows = any_worst_df[any_worst_df['SPEI'] == False]

# Compare lengths and indices
match_count = len(nan_quality_rows) == len(spei_false_rows)

print(f"Number of NaN Quality rows: {len(nan_quality_rows)}")
print(f"Number of SPEI=False rows: {len(spei_false_rows)}")
print(f"Do the indices match exactly? {match_count}")

if match_count:
    print("\nConfirmation: The rows with missing physical features are a 100% match for SDO Earth Shadow events.")

Number of NaN Quality rows: 17
Number of SPEI=False rows: 21
Do the indices match exactly? False
